# 1 — From Data to Model: The Core Braindecode Pipeline

**Cutting-EEG Workshop 2026 · UCSD · Deep Learning EEG Methods and Practice**

Most EEG researchers already know how to preprocess data. The wall people hit is the
next step: turning a preprocessed recording into something PyTorch will actually train
on. This notebook is about that wall.

By the end you will have a **template you can paste into your own lab's code** and point
at your own data.

### What we do here

1. Load BCI Competition IV 2a (motor imagery) through Braindecode's MOABB interface
2. Preprocess it — the same steps you'd run in EEGLAB, expressed as a pipeline
3. Cut continuous data into **windows**, the unit a neural net consumes
4. Split by session, respecting the train/test protocol the dataset defines
5. Train `EEGNet` with a plain, readable PyTorch loop
6. Look at what the model learned, and where it fails

### The one idea to take away

> A deep learning EEG pipeline is `Raw → Preprocess → Window → Dataset → DataLoader → Model`.
> Braindecode gives you a well-tested implementation of each arrow. Everything else in
> this workshop is a variation on this chain.

---
**Runtime: `Runtime → Change runtime type → T4 GPU`.** It works on CPU, just slower.

## 0 · Setup

Two things worth knowing before you run this:

- `braindecode[moabb]` — the `[moabb]` part matters. Plain `pip install braindecode`
  will not give you the dataset downloader, and you'll get an ImportError later.
- **The runtime restart is not optional.** The install upgrades numpy, pandas, and mne
  past the versions Colab preloaded. Python has already imported the old C extensions,
  and they will crash on the new ones. The cell restarts the kernel for you.

Run the next cell, wait for the restart notice, then continue from Section 1. You do
**not** re-run this cell after the restart.

In [ ]:
%pip install -q "braindecode[moabb]"

# Restart so the freshly installed numpy/mne are the ones actually loaded.
import IPython
print("Install finished — restarting the runtime.")
print("This 'crash' notice is expected. Continue at Section 1 below.")
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Verify the environment. Run this AFTER the restart above.
import numpy as np
import torch
import mne
import braindecode

mne.set_log_level("ERROR")  # MNE is chatty; we only want our own output

print(f"braindecode {braindecode.__version__}")
print(f"torch       {torch.__version__}")
print(f"mne         {mne.__version__}")
print(f"numpy       {np.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {DEVICE}")
if DEVICE == "cpu":
    print("  (No GPU. Everything still runs — training takes a few minutes longer.)")

SEED = 20260916
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1 · Loading data

We use **BCI Competition IV dataset 2a**: 9 subjects, 22 EEG channels, 250 Hz, four
motor imagery classes (left hand, right hand, feet, tongue). It is the *fruit fly* of
EEG decoding — small, public, and every method in the literature reports on it, so you
can immediately tell whether your number is reasonable.

`MOABBDataset` handles download and caching. First run pulls ~40 MB per subject.

### Using your own data instead

This is the only cell you'd replace. Braindecode accepts any list of `mne.io.Raw`:

```python
from braindecode.datasets import create_from_mne_raw
raw = mne.io.read_raw_eeglab("my_data.set", preload=True)   # EEGLAB .set
dataset = create_from_mne_raw([raw], ...)
```

MNE reads EEGLAB `.set`, BrainVision `.vhdr`, EDF, FIF, and more. Once you have a
`Raw`, everything downstream in this notebook is unchanged.

In [ ]:
from braindecode.datasets import MOABBDataset

SUBJECT_ID = 3  # a middling subject — not the easiest, not the hardest

# Note: "BNCI2014_001" with the underscore. MOABB renamed its datasets in v1.0;
# the old "BNCI2014001" still resolves but emits a deprecation warning.
dataset = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=[SUBJECT_ID])

print(f"Loaded subject {SUBJECT_ID}: {len(dataset.datasets)} recordings\n")
print(dataset.description)

### What did we just load?

A `BaseConcatDataset` — a list of recordings plus a `description` table of metadata.
The two rows are the two **sessions**, recorded on different days. That detail drives
how we split later.

Let's look at the raw signal underneath.

In [ ]:
raw = dataset.datasets[0].raw
print(f"Channels    : {len(raw.ch_names)}")
print(f"Sampling    : {raw.info['sfreq']} Hz")
print(f"Duration    : {raw.n_times / raw.info['sfreq']:.0f} s")
print(f"Channel names: {raw.ch_names[:8]} ...")

# The event annotations mark when each motor imagery trial begins.
import pandas as pd
events = pd.Series([a["description"] for a in raw.annotations])
print(f"\nTrial counts in this session:\n{events.value_counts().to_string()}")

## 2 · Preprocessing

Four steps, each a `Preprocessor` in a list applied in order. Nothing here is specific
to deep learning — you would do the same in EEGLAB. The difference is that this is a
*declarative pipeline*, so it is reproducible and re-runnable.

| Step | Why |
|---|---|
| Pick EEG channels | Drop the EOG channels; keep the 22 EEG ones |
| Scale to µV | Convert from volts. Raw volts are ~1e-6 — too small for stable gradients |
| Bandpass 4–38 Hz | Motor imagery lives in mu (~8–12) and beta (~13–30). Also kills line noise and drift |
| Exponential moving standardize | Running z-score per channel — handles the slow drift EEG amplitude has over a session |

That last one is worth dwelling on. A fixed z-score over the whole recording assumes
the signal is stationary. EEG is not: impedance changes, the subject gets tired.
Exponential moving standardization tracks a running mean/variance instead, so a window
at minute 40 is normalized against its local context rather than the session average.

**Filtering choices are a modelling decision, not a formality.** Cut at 38 Hz and you
have decided gamma carries nothing for your task. That's defensible for motor imagery
and wrong for some other paradigms.

In [ ]:
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    exponential_moving_standardize,
)

LOW_HZ, HIGH_HZ = 4.0, 38.0
FACTOR_NEW, INIT_BLOCK = 1e-3, 1000

preprocessors = [
    Preprocessor("pick_types", eeg=True, meg=False, eog=False),
    Preprocessor(lambda data: data * 1e6),                       # V → µV
    Preprocessor("filter", l_freq=LOW_HZ, h_freq=HIGH_HZ),
    Preprocessor(
        exponential_moving_standardize,
        factor_new=FACTOR_NEW,
        init_block_size=INIT_BLOCK,
    ),
]

preprocess(dataset, preprocessors, n_jobs=1)
print("Preprocessing complete.")

sfreq = dataset.datasets[0].raw.info["sfreq"]
n_channels = len(dataset.datasets[0].raw.ch_names)
print(f"{n_channels} channels @ {sfreq} Hz")

## 3 · Windowing — the step that trips people up

A neural network needs fixed-size tensors. Continuous EEG is one long stream. Windowing
bridges the two, and **the window definition encodes a hypothesis about your task.**

For this dataset the cue appears at t=0 and the subject imagines movement for 4 seconds.
We take 0.5 s *before* the cue through the end of the trial. Why start early? The
pre-cue baseline gives the network a reference for what "no imagery" looks like in that
subject at that moment.

`trial_start_offset_samples` is in **samples, not seconds** — a classic off-by-250 bug.
We compute it from `sfreq` so it stays correct if you change datasets.

In [ ]:
from braindecode.preprocessing import create_windows_from_events

TRIAL_START_OFFSET_SEC = -0.5
trial_start_offset_samples = int(TRIAL_START_OFFSET_SEC * sfreq)

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=trial_start_offset_samples,
    trial_stop_offset_samples=0,
    preload=True,
)

X, y, _ = windows_dataset[0]
print(f"One window : {X.shape}  -> (channels, timepoints)")
print(f"Its label  : {y}")
print(f"Total windows: {len(windows_dataset)}")

n_times = X.shape[1]
print(f"\nWindow length: {n_times} samples = {n_times / sfreq:.1f} s")

### Note the shape: `(channels, time)`

Braindecode's convention is `(batch, channels, time)` — EEG as a multichannel 1-D
signal. This differs from image conventions and from what some other EEG toolboxes use.
When a model errors with a shape mismatch, this is the first thing to check.

In [ ]:
# Class names, and whether the classes are balanced.
import collections

labels = [windows_dataset[i][1] for i in range(len(windows_dataset))]
class_names = windows_dataset.datasets[0].windows.event_id
print(f"Class mapping: {class_names}")
print(f"Distribution : {dict(sorted(collections.Counter(labels).items()))}")
print("\nBalanced — so plain accuracy is a fair metric here.")
print("On your own data, check this before trusting accuracy.")

## 4 · Splitting — the most common way to fool yourself

**Do not split EEG windows randomly.** Adjacent windows share drift, electrode state,
and subject alertness. A random split leaks that shared context into the test set and
your accuracy is inflated — sometimes by 20 points. This is the single most frequent
error in EEG deep learning papers.

BCI IV 2a was recorded as two sessions on different days, precisely so you can evaluate
across a realistic gap. We honour that: session 1 trains, session 2 tests.

The session keys are `"0train"` / `"1test"`. MOABB 1.0 renamed these from
`"session_T"` / `"session_E"`, so older tutorials will fail here. We print the keys
rather than assuming — a habit worth keeping.

In [ ]:
splits = windows_dataset.split("session")
print(f"Available session keys: {list(splits.keys())}")

train_set = splits["0train"]
test_set = splits["1test"]

print(f"\nTrain (session 1): {len(train_set)} windows")
print(f"Test  (session 2): {len(test_set)} windows")

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

# Braindecode datasets yield THREE items: (X, y, crop_indices).
# A standard `for X, y in loader` loop raises "too many values to unpack" here.
# This is the most common integration bug when moving from a torchvision tutorial.
batch = next(iter(train_loader))
print(f"Items per batch: {len(batch)}  (X, y, crop_inds)")
print(f"X: {batch[0].shape}  dtype={batch[0].dtype}")
print(f"y: {batch[1].shape}  dtype={batch[1].dtype}")

## 5 · The model

`EEGNet` is the standard baseline for EEG classification — a compact CNN whose
structure mirrors what EEG researchers do by hand:

1. **Temporal convolution** — learns frequency filters. This is a learned bandpass.
2. **Depthwise spatial convolution** — learns a weighting across channels. This is
   learned spatial filtering, the same job CSP does, but optimized for the task.
3. **Separable convolution** — combines the two into features.

That is why EEGNet works with ~2,000 parameters where a generic CNN would need
millions. The architecture already knows EEG is `(channels × time)` with spatial
structure across channels and oscillatory structure across time.

> **API note.** In Braindecode ≥1.7 this class is `EEGNet`, not `EEGNetv4`, and the
> arguments are `n_chans` / `n_outputs` / `n_times`. The older
> `in_chans` / `n_classes` / `input_window_samples` names were **removed**, not
> deprecated — they now raise. Tutorials written before 2025 will not run as-is.

In [ ]:
from braindecode.models import EEGNet

n_classes = len(class_names)

model = EEGNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    final_conv_length="auto",
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"EEGNet: {n_params:,} trainable parameters")
print(f"Input  (batch, {n_channels}, {n_times}) -> output (batch, {n_classes})\n")

with torch.no_grad():
    dummy = torch.randn(2, n_channels, n_times, device=DEVICE)
    print(f"Shape check: {tuple(dummy.shape)} -> {tuple(model(dummy).shape)}")

## 6 · The training loop

Deliberately written as plain PyTorch. Braindecode ships `EEGClassifier` (a scikit-learn
style wrapper) and it is genuinely convenient — but a visible loop is what you need when
you have to modify something, and modifying something is what research is.

Two EEG-specific choices:

- **AdamW with weight decay.** EEG datasets are small (here: ~280 training trials for
  ~2k parameters). Regularization is not optional.
- **Cosine annealing.** Smoothly decays the learning rate. Reliably worth a point or
  two over a constant rate, for one line of code.

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

N_EPOCHS = 60
LR = 6e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)


def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    """One pass over `loader`. Trains if `optimizer` is given, else evaluates."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_correct, n_total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for X, y, _ in loader:                    # note the third element
            X, y = X.to(device).float(), y.to(device).long()

            logits = model(X)
            loss = criterion(logits, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(y)
            n_correct += (logits.argmax(dim=1) == y).sum().item()
            n_total += len(y)

    return total_loss / n_total, n_correct / n_total

In [ ]:
import time

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
start = time.time()

print(f"Training for {N_EPOCHS} epochs on {DEVICE}\n")
for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    te_loss, te_acc = run_epoch(model, test_loader, criterion, None)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["test_loss"].append(te_loss)
    history["test_acc"].append(te_acc)

    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch {epoch:3d}/{N_EPOCHS} | "
              f"train {tr_loss:.3f}/{tr_acc:.1%} | "
              f"test {te_loss:.3f}/{te_acc:.1%}")

print(f"\nDone in {time.time() - start:.0f}s")
print(f"Final test accuracy: {history['test_acc'][-1]:.1%}")
print(f"Best  test accuracy: {max(history['test_acc']):.1%}  (chance = {1/n_classes:.0%})")

## 7 · Reading the curves

Plot both curves together. The gap between them tells you what to do next, and it is
almost always more informative than the final number.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
epochs = range(1, N_EPOCHS + 1)

ax1.plot(epochs, history["train_loss"], label="train", lw=2)
ax1.plot(epochs, history["test_loss"], label="test", lw=2)
ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy loss")
ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history["train_acc"], label="train", lw=2)
ax2.plot(epochs, history["test_acc"], label="test", lw=2)
ax2.axhline(1 / n_classes, ls="--", c="gray", label="chance")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy")
ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f"EEGNet · BCI IV 2a subject {SUBJECT_ID}", y=1.02)
plt.tight_layout(); plt.show()

gap = history["train_acc"][-1] - history["test_acc"][-1]
print(f"Train − test gap: {gap:.1%}")
print("  >20% → overfitting. More dropout, more weight decay, or augmentation.")
print("  Both curves low → underfitting. Train longer, or raise the learning rate.")

## 8 · Where does it fail?

Accuracy is one number summarizing 288 decisions. The confusion matrix shows you *which*
decisions, and for motor imagery the error structure is interpretable: hand-vs-hand
confusions mean the spatial filter isn't separating left from right motor cortex, which
is a different problem from feet-vs-tongue confusion.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X, y, _ in test_loader:
        preds = model(X.to(DEVICE).float()).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_true.append(y)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).numpy()

names = [k for k, _ in sorted(class_names.items(), key=lambda kv: kv[1])]
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(5.5, 4.8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Confusion matrix")
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=names, digits=3))

## 9 · Swapping the model

The payoff of the pipeline shape: the model is one line. Everything else is unchanged.
`ShallowFBCSPNet` is a deep-learning reimplementation of the classic FBCSP algorithm —
the method that won the original BCI competition — so this is a genuine
old-school-vs-new-school comparison.

In [ ]:
from braindecode.models import ShallowFBCSPNet

model2 = ShallowFBCSPNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    final_conv_length="auto",
).to(DEVICE)

opt2 = AdamW(model2.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched2 = CosineAnnealingLR(opt2, T_max=N_EPOCHS)

print(f"ShallowFBCSPNet: {sum(p.numel() for p in model2.parameters()):,} parameters")
print(f"(EEGNet had {n_params:,})\n")

acc2 = []
for epoch in range(1, N_EPOCHS + 1):
    run_epoch(model2, train_loader, criterion, opt2)
    _, te_acc = run_epoch(model2, test_loader, criterion, None)
    sched2.step()
    acc2.append(te_acc)
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | test {te_acc:.1%}")

print(f"\n{'Model':<18}{'best test acc':>14}")
print(f"{'EEGNet':<18}{max(history['test_acc']):>13.1%}")
print(f"{'ShallowFBCSPNet':<18}{max(acc2):>13.1%}")

### Interpreting the comparison

On a single subject with ~280 training trials, these two typically land within a few
points of each other, and **which one wins varies by subject and by random seed**.

That is itself the lesson. A 2-point difference on one subject is noise. Before you
believe an architecture is better, you want multiple subjects and multiple seeds. This
is where a lot of published EEG comparisons are weaker than they look.

## 10 · The template

Here is the whole pipeline, compressed. This is the piece to take home: change the
loading cell to point at your data, change `n_outputs` to your class count, and the rest
holds.

In [ ]:
TEMPLATE = '''
# ---- Braindecode pipeline template -------------------------------------
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor, preprocess, exponential_moving_standardize,
    create_windows_from_events,
)
from braindecode.models import EEGNet
from torch.utils.data import DataLoader

# 1. LOAD — swap for create_from_mne_raw([your_raw]) to use your own data
dataset = MOABBDataset("BNCI2014_001", subject_ids=[3])

# 2. PREPROCESS
preprocess(dataset, [
    Preprocessor("pick_types", eeg=True, meg=False, eog=False),
    Preprocessor(lambda d: d * 1e6),
    Preprocessor("filter", l_freq=4.0, h_freq=38.0),
    Preprocessor(exponential_moving_standardize,
                 factor_new=1e-3, init_block_size=1000),
])

# 3. WINDOW — offset is in SAMPLES
sfreq = dataset.datasets[0].raw.info["sfreq"]
windows = create_windows_from_events(
    dataset, trial_start_offset_samples=int(-0.5 * sfreq),
    trial_stop_offset_samples=0, preload=True)

# 4. SPLIT — by session/subject, never randomly
splits = windows.split("session")
train_set, test_set = splits["0train"], splits["1test"]

# 5. LOAD IN BATCHES
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, drop_last=True)
test_loader  = DataLoader(test_set,  batch_size=64)

# 6. MODEL
X, y, _ = train_set[0]
model = EEGNet(n_chans=X.shape[0], n_outputs=4,
               n_times=X.shape[1], final_conv_length="auto")

# 7. TRAIN — batches yield THREE items: for X, y, _ in train_loader
# ------------------------------------------------------------------------
'''
print(TEMPLATE)

## Recap

You built a complete EEG deep learning pipeline and, more importantly, saw the four
places it usually goes wrong:

| Trap | Fix |
|---|---|
| `pip install braindecode` without `[moabb]` | Install the extra; restart the runtime |
| Random train/test split | Split by session or subject |
| `for X, y in loader` | Braindecode yields `(X, y, crop_inds)` |
| `in_chans` / `n_classes` from an old tutorial | Now `n_chans` / `n_outputs` / `n_times` |

### Try it yourself

1. Change `SUBJECT_ID` to 1, then 6. Between-subject variance on this dataset is large —
   this is the central difficulty of EEG decoding, and it motivates Notebook 3.
2. Widen the bandpass to 4–120 Hz. Does more spectrum help, or just add noise?
3. Set `trial_start_offset_samples=0` to drop the pre-cue baseline. How much does it matter?

**Next:** Notebook 2 replaces the CNN with a Transformer, and asks what attention buys
you on signals this small.